# Hate Speech Detection (Twitter)

Classifies tweets into **hate speech**, **offensive language**, or **neither** using classic NLP + machine learning (Bag-of-Words, scikit-learn).

> **Content warning:** the dataset contains slurs and offensive language.

**Pipeline:** load data → clean text → train/test split → compare models → evaluate → predict on new text.

## 1. Imports

In [ ]:
import re
import string

import joblib
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
nltk.download("stopwords", quiet=True)

## 2. Load data

In [ ]:
DATA_PATH = "data/twitter.csv"  # change if your file is elsewhere

dataset = pd.read_csv(DATA_PATH)
dataset.head()

In [ ]:
dataset.info()
print("\nMissing values per column:")
print(dataset.isnull().sum())

The `class` column holds the label: 0 = hate speech, 1 = offensive language, 2 = neither.

In [ ]:
label_map = {0: "hate speech", 1: "offensive language", 2: "no hate speech or offensive language"}
dataset["labels"] = dataset["class"].map(label_map)

data = dataset[["tweet", "labels"]].copy()
data["labels"].value_counts()

The classes are **heavily imbalanced** (hate speech is only about 5% of the data), so accuracy alone is misleading. We use macro-F1 and per-class metrics below.

In [ ]:
data["labels"].value_counts().plot(kind="bar", rot=15, title="Class distribution")
plt.ylabel("Tweets")
plt.tight_layout()
plt.show()

## 3. Text cleaning

In [ ]:
stop_words = set(stopwords.words("english"))
stemmer = SnowballStemmer("english")
PUNCT_RE = re.compile("[%s]" % re.escape(string.punctuation))


def clean_data(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", "", text)  # URLs
    text = re.sub(r"@\w+", "", text)                    # @mentions
    text = re.sub(r"&\w+;|&#\d+;", " ", text)           # HTML entities (&amp; etc.)
    text = re.sub(r"\brt\b", "", text)                  # retweet marker
    text = re.sub(r"\[.*?\]", "", text)                 # [bracketed text]
    text = re.sub(r"<.*?>", "", text)                    # HTML tags
    text = PUNCT_RE.sub("", text)                        # punctuation
    text = re.sub(r"\n", " ", text)
    text = re.sub(r"\w*\d\w*", "", text)                # words containing digits
    words = [w for w in text.split() if w not in stop_words]
    return " ".join(stemmer.stem(w) for w in words)


data["tweet"] = data["tweet"].apply(clean_data)
data.head()

## 4. Train/test split
Stratified, so each split keeps the same class proportions.

In [ ]:
X = data["tweet"]
y = data["labels"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

## 5. Model comparison
Each model is a `Pipeline` (vectorizer + classifier), so the `CountVectorizer` is fit **only on training data** (no data leakage).

In [ ]:
models = {
    "Decision Tree (baseline)": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Decision Tree (balanced)": DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
    "Logistic Regression (balanced)": LogisticRegression(class_weight="balanced", max_iter=1000),
    "Multinomial Naive Bayes": MultinomialNB(),
}

fitted, rows = {}, []
for name, clf in models.items():
    pipe = Pipeline([("vec", CountVectorizer()), ("clf", clf)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted[name] = pipe
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "macro_f1": f1_score(y_test, pred, average="macro"),
    })

results = pd.DataFrame(rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
results.round(4)

## 6. Evaluate the best model (by macro-F1)

In [ ]:
best_name = results.loc[0, "model"]
best_model = fitted[best_name]
y_pred = best_model.predict(X_test)
print("Best model:", best_name, "\n")
print(classification_report(y_test, y_pred))

In [ ]:
class_order = list(label_map.values())
cm = confusion_matrix(y_test, y_pred, labels=class_order)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="YlGnBu",
            xticklabels=["hate", "offensive", "neither"],
            yticklabels=["hate", "offensive", "neither"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion matrix: {best_name}")
plt.tight_layout()
plt.show()

## 7. Predict on new text

In [ ]:
def predict_text(text, model=best_model):
    return model.predict([clean_data(text)])[0]


samples = [
    "i am going to kill that bitch",
    "have a great day everyone",
    "this movie was trash",
]
for s in samples:
    print(f"{s!r:45} -> {predict_text(s)}")

## 8. Save the model (optional)

In [ ]:
joblib.dump(best_model, "hate_speech_model.joblib")
# Load later with: model = joblib.load("hate_speech_model.joblib")

## Limitations
- Bag-of-Words ignores word order and context (sarcasm, reclaimed slurs, quoted speech).
- Hate speech is a small minority class, so its precision/recall is much lower than the other classes.
- The dataset is from 2017 and English-only; language and slang drift will hurt real-world performance.
- Labels are crowd-sourced and noisy. Do not use this model for automated moderation without human review.